In [15]:
# 준비: Lab 6에서 가져온 모델·토크나이저 로드
import torch
import textwrap
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map="auto"
)
model.eval()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

In [16]:
import torch

def chat(model,
         tokenizer,
         user_message,
         system_message="You are a helpful assistan",
         **generation_args):

    # 1. 역할별 메시지 구성
    # ssytem : 모델의 성격과 규칙, user : "사람의 질문", assistant : 이전 모델 답변
    # 왜 assistant 역할이 필요할까? -> LLM은 기억력이 없음 그래서 매 요청 마다 이전의 대화를 전부 다시 넣어줘야함!
    messages = [
        {"role" : "system", "content" :system_message},
        {"role": "user", "content": user_message},
    ]

    # 2. Qwen이 학습할 때 쓴 형식으로 변환하기
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False, # 토큰을 반환하는 것이 아니라, 사람이 읽을 수 있는 텍스트를 반환
        add_generation_prompt=True # assistant의 차례를 명시하는 부분
    )

    # 3. 문자열을 토큰ID 텐서로 바꾸고, gpu로 이동 ( 모델이 있는 곳 )
    inputs = tokenizer(
        text,
    return_tensors="pt").to(model.device)

    # 4. 추론이기 때문에 gradient 계산 삭제하기 ( 메모리 절약 )
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            **generation_args
        )

    # 5. generate는 프롬프트 + 답변을 같이 돌려줌 -> 프롬프트 길이 만큼 잘라내서 답변만 출력함
    new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]

    # 6. 토큰 ID -> 문자열, <|im_end|> 같은 특수 토큰을 제거
    return tokenizer.decode(
        new_tokens,
        skip_special_tokens = True
    )

In [17]:
# 디코딩 옵션 비교

# Greedy : 매번 확률 1등 토큰만 선택하도록 -> 항상 같은 답이 선택이 됨!
response = chat(model,
                tokenizer,
                "Recommend a Thai restaurant meal.",
                max_new_tokens=80,
                do_sample=False)
print("[Greedy]")
print(textwrap.fill(response, width=80))

# Sampling : 매번 다른 답을 낼 수 있음, temperature(분포를 날카롭게/평평하게), top_k(상위 k개만 후보), top_p(누적확률 p까지만 후보)로 다양성을 조절 가능
response = chat(model,
                tokenizer,
                "Suggest a dinner menu.",
                max_new_tokens=80,
                do_sample=True,
                temperature=0.8,
                top_k=20)
print("\n[Sampling]")
print(textwrap.fill(response, width=80))

[Greedy]
Here's a recommendation for a Thai restaurant meal:  Pad Thai with Tom Yum Soup
This dish is a classic combination of Thai cuisine that you won't want to miss.
The Pad Thai, which translates to "sticky rice noodles," features stir-fried
rice noodles in a flavorful sauce made from tamarind paste, fish sauce, and lime
juice. It's typically served with shrimp or tofu,

[Sampling]
Here's a suggestion for a delicious and flavorful dinner menu:  1. Start with an
appetizer: Caprese salad with fresh mozzarella, tomatoes, basil, and balsamic
glaze. 2. Main course: Grilled salmon with lemon butter sauce, asparagus, and
roasted potatoes. 3. Side dish: Roasted garlic mashed cauliflower. 4. Salad:
Mixed greens with cherry tomatoes,


In [18]:
# Qwen 챗봇 만들기

import ipywidgets as widgets
from IPython.display import display
import textwrap

history = []   # (user-사용자의 질문, assistant-모델의 답변) 튜플을 쌓아두는 대화 기록

user_input = widgets.Text(placeholder="Type your message here...",
                          description="You:",
                          layout=widgets.Layout(width="80%")
            )
send_button = widgets.Button(description="Send", button_style="primary")
chat_output = widgets.Output()

def send_message(_):
    message = user_input.value.strip() # 메시지의 공백을 제거

    if not message:
        return

    # 입력창 비우기
    user_input.value = ""

    # 프롬프트 생성하기, 최근 5번의 대화 기록과 이번 질문을 더하기
    prompt = ""
    for user, assistant in history[-5:]: # 질문, 답변 -> 튜플을 꺼내기
        prompt += f"User: {user}\nAssistant: {assistant}\n"
    prompt += f"User: {message}\nAssistant:" # 마직막은 Assistance: 로 끝내서 모델이 이어서 답하기 유도하기

    with chat_output:
        print(f"\nYou: {message}")

    # 위에서 생성한 프롬프트를 유저 메시지에 넣기
    response = chat(model, tokenizer, prompt, max_new_tokens=80,
                    do_sample=True, temperature=0.8, top_k=20, top_p=0.9)

    # 나눈 대화를 기록에 추가, 다음 질문에서 사용
    history.append((message, response))

    with chat_output:
        print("Assistant:")
        print(textwrap.fill(response, width=80))

send_button.on_click(send_message)   # 버튼 클릭 이벤트 연결
display(widgets.VBox([chat_output, widgets.HBox([user_input, send_button])]))

In [19]:
!pip install -q peft bitsandbytes
!pip uninstall -y -q torchao   # 코랩 기본 torchao(0.10)와 최신 peft 충돌 방지 (Lab 7에서는 안 씀)

In [20]:
# LoRA 실습 공통 설정
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model # LoRA 설정·부착

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

# SFT 학습 데이터 준비 : (사용자 입력, 모델이 배워야 할 응답) - 10개 세트
# 목적 : 고품질 맛집 모델이 아니라, "음식 종류·가격·위치 중 빠진 정보가 있으면 비어 있는 것에 대해 먼저 되묻고, 다 있으면 검색을 진행한다는 행동 규칙 세우기
'''
SFT(Supervised Fine-Tuning) Pre-training 이후의 언어모델에‘질문, 지시(입력)와 모범 답안(출력)’ 쌍으로 된 데이터를 학습시켜 사용자의 의도에 맞게 응답 및 동작하도록 만드는 파인튜닝 기법을 의미한다!
'''
training_data = [
    # 빠진 정보가 있음 -> 되묻기 (6개)
    ("I want Thai food under ten dollars.",
     "What location would you prefer for the Thai restaurant?"),        # 위치 X
    ("I want Chinese food downtown.",
     "What price range would you prefer?"),                             # 가격 X
    ("I need cheap Korean food.",
     "What location would you prefer for the Korean restaurant?"),      # 위치 X
    ("Find an Italian restaurant near campus.",
     "What price range would you prefer?"),                             # 가격 X
    ("I want Indian food but I do not know the price.",
     "What price range would you prefer?"),                             # 가격과 위치 X → 하나씩 질문
    ("Find Japanese food under fifteen dollars.",
     "What location would you prefer?"),                                # 위치 X

    # 정보가 다 있음 -> 검색 진행 (4개)
    ("I want inexpensive Japanese food downtown.",
     "I will look for an inexpensive Japanese restaurant downtown."),
    ("I want a cheap Thai restaurant downtown.",
     "I will look for a cheap Thai restaurant downtown."),
    ("Find expensive Italian food nearby.",
     "I will look for an expensive Italian restaurant nearby."),
    ("I want Korean food near campus under twenty dollars.",
     "I will look for a Korean restaurant near campus under twenty dollars."),
]

# 길이가 제각각인 문장의 뒤를 채울 토큰을 정하는 부분
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

SYSTEM_MESSAGE = "You are a concise restaurant assistant."
MAX_LENGTH = 128   # 모든 학습 문장을 128 토큰으로 통일

class RestaurantDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.examples = []

        for user_text, assistant_text in data:
            # 1. 프롬프트(system + user)와 전체 대화(+ 정답 응답) 메시지 구성
            prompt_messages = [
                {"role": "system", "content": SYSTEM_MESSAGE},
                {"role": "user", "content": user_text}
            ]
            full_messages = prompt_messages + [ # 슬라이드에서 생략된 부분
                {"role": "assistant", "content": assistant_text}
            ]

            # 2. 프롬프트만 : "<|im_start|>assistant"까지 -> 어디까지 가릴지 길이를 카운트
            # 문제가 어디에서 끝나는지!
            prompt_text = tokenizer.apply_chat_template(
                prompt_messages,
                tokenize=False,
                add_generation_prompt=True
            )
            # 3. 전체 학습 시퀀스 : 문제 + 정답이 같이 있는 교사용 문제지 같은 것 ( 정답 + <im_end>까지 )
            # 실제로 모델에게 보여 주는 학습 입력
            # 앞부분은 Prompt와 동일, 그래서 Prompt 부분 이후만 채점
            full_text = tokenizer.apply_chat_template(
                full_messages,
                tokenize=False,
                add_generation_prompt=False
            )
            # 둘의 차이 : Prompt는 정답 직전까지(경계선 재기용), Full text는 정답까지 포함(실제 학습 입력) => 둘의 길이 차이 = 채점할 정답 구간

            # 4. 프롬프트 길이 측정 (템플릿이 특수 토큰을 이미 넣었으므로 중복 추가 X)
            prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]

            # 5. 전체 시퀀스 토큰화 + 128칸으로 맞추기 (짧으면 pad, 길면 자름)
            encoding = tokenizer(
                full_text,
                add_special_tokens=False,
                max_length=MAX_LENGTH,
                truncation=True,
                padding="max_length",
                return_tensors="pt"
            )
            input_ids = encoding["input_ids"].squeeze(0) # [1, 128] -> [128]
            attention_mask = encoding["attention_mask"].squeeze(0)  # [1, 128] -> [128]

            # 6. labels : assistant 응답만 채점하도록 나머지는 -100
            labels = input_ids.clone()
            prompt_length = min(len(prompt_ids), MAX_LENGTH)
            labels[:prompt_length] = -100 # system + user => 문제(Prompt 길이만큼)는 채점 X => 정답 구간만 채점
            labels[attention_mask == 0] = -100 # 패딩 채점 X

            self.examples.append({
                "input_ids": input_ids,
                "attention_mask": attention_mask,
                "labels": labels
            })

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        return self.examples[index]


dataset = RestaurantDataset(
    training_data,
    tokenizer
)
loader = DataLoader(
    dataset,
    batch_size=1,
    shuffle=True
)

In [21]:
# 학습 함수: LoRA 어댑터만 학습

def train_adapter(model,
                  loader,
                  epochs=3,
                  learning_rate=2e-4
    ):

    # requires_grad=True 인 파라미터(= LoRA 어댑터)만 옵티마이저에 등록
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=learning_rate
    )
    model.config.use_cache = False   # 학습 중에는 KV Cache 끄기
    model.train()

    for epoch in range(epochs):
        total_loss = 0
        for batch in loader:
            batch = {key: value.to(model.device) for key, value in batch.items()}

            optimizer.zero_grad() # 기울기 초기화
            output = model(**batch) # 순전파
            loss = output.loss # assistant 부분 만 loss 계산
            loss.backward() # 역전파
            optimizer.step() # LoRA 가중치를 실제로 고치기
            total_loss += loss.item()
        average_loss = total_loss / len(loader)
        print(f"Epoch {epoch+1}: Loss = {average_loss:.4f}")

    model.config.use_cache = True    # 학습 끝나면 다시 켜서 generate() 빠르게

# 기존의 구조와 동일하지만 학습되는 파라미터가 어뎁터뿐 이라는 점만 다름!!

In [22]:
# 파라미터 개수와 GPU 메모리 사용량 확인하는 함수 생성!

# 파라미터 개수 구하기
def count_parameters(model):
    total = sum(p.numel() for p in model.parameters()) # 전체 파라미터 수 구하기
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad) # 학습 가능한 requires_grad True를 구하기, LoRA에 해당됨!

    print(f"Total parameters: {total:,}")
    print(f"Trainable parameters: {trainable:,}")
    print(f"Trainable %: {100*trainable/total:.4f}")

# GPU 메모리 사용량 확인하기
def show_gpu_memory():
    if not torch.cuda.is_available():
        print("GPU를 사용할 수 없습니다 (CPU 런타임).")   # 조용히 끝나지 않고 이유 출력
        return
    torch.cuda.synchronize()
    allocated = torch.cuda.memory_allocated() / 1024**3 # PyTorch 텐서가 차지한 메모리를 GB로 변환하기
    peak = torch.cuda.max_memory_allocated() / 1024**3 # 가장 피크였던 메모리의 사용량

    print(f"Allocated: {allocated:.2f} GB")
    print(f"Peak: {peak:.2f} GB")

count_parameters(model)
show_gpu_memory()

Total parameters: 1,543,714,304
Trainable parameters: 1,543,714,304
Trainable %: 100.0000
Allocated: 5.76 GB
Peak: 5.77 GB


In [26]:
# LoRA 어뎁터만 추가하기

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
count_parameters(model)


train_adapter(model, loader, epochs=3, learning_rate=2e-4)
show_gpu_memory()

# 미세조정 모델 (어댑터 ON)
response = chat(model, tokenizer, "I want cheap Thai food.",
                system_message=SYSTEM_MESSAGE, max_new_tokens=60, do_sample=False)
print("어뎁터 ON") #
print(response)

# 원본 모델 (어댑터 OFF)
with model.disable_adapter():
    response = chat(model, tokenizer, "I want cheap Thai food.",
                    system_message=SYSTEM_MESSAGE, max_new_tokens=80, do_sample=False)
    print("어뎁터 OFF")
    print(response)

trainable params: 1,089,536 || all params: 1,544,803,840 || trainable%: 0.0705
Total parameters: 1,544,803,840
Trainable parameters: 1,089,536
Trainable %: 0.0705
Epoch 1: Loss = 2.1297
Epoch 2: Loss = 1.1983
Epoch 3: Loss = 0.6859
Allocated: 5.77 GB
Peak: 6.35 GB
어뎁터 ON
What would you like to drink?
어뎁터 OFF
Here are some affordable Thai restaurants in various cities:

1. **New York City:**
   - **Thai House** (20 East 4th Street, Manhattan) - Offers authentic Thai cuisine at reasonable prices.
   - **Thaipassion** (35 West 8th Street, Greenwich Village) - Known for its spicy curries and noodles.

2. **Los Angeles


In [27]:
# LoRA 어댑터만 저장
model.save_pretrained("./qwen_restaurant_lora")